In [1]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

mesh_path = r"C:\Eli Folder temp\geotransformer-faces-updated\scans\FaMoS_180424_03335_TA\natural_head_rotation.000001.obj"
landmark_path = r"data/faces/demo/ref_3.npy"

scale_factor = 1.0 # mesh scaling only

# ---- Mesh to scaled point cloud ----
mesh = o3d.io.read_triangle_mesh(mesh_path)
mesh.compute_vertex_normals()
pcd = mesh.sample_points_uniformly(number_of_points=20000)

mesh_points = np.asarray(pcd.points) * scale_factor


# ---- Center mesh at (0,0,0) ----
centroid = mesh_points.mean(axis=0)
mesh_points = mesh_points - centroid

# ---- Load landmark / ref points (UNSCALED) ----
ref_points = np.load(landmark_path)

# ---- Combine for proper axis scaling ----
all_points = np.vstack([mesh_points, ref_points])

min_bound = all_points.min(axis=0)
max_bound = all_points.max(axis=0)
center = (min_bound + max_bound) / 2.0
extent = (max_bound - min_bound).max() / 2.0

x_range = [center[0] - extent, center[0] + extent]
y_range = [center[1] - extent, center[1] + extent]
z_range = [center[2] - extent, center[2] + extent]

# ---- Plot ----
fig = go.Figure()

# Scaled mesh
fig.add_trace(go.Scatter3d(
    x=mesh_points[:, 0],
    y=mesh_points[:, 1],
    z=mesh_points[:, 2],
    mode='markers',
    marker=dict(size=2),
    name="Scaled Mesh"
))
"""
# Unscaled reference points
fig.add_trace(go.Scatter3d(
    x=ref_points[:, 0],
    y=ref_points[:, 1],
    z=ref_points[:, 2],
    mode='markers',
    marker=dict(size=6),
    name="Reference Points"
))
"""
fig.update_layout(
    scene=dict(
        xaxis=dict(range=x_range),
        yaxis=dict(range=y_range),
        zaxis=dict(range=z_range),
        aspectmode='cube'
    )
)

fig.show()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
np.save("plank.npy", mesh_points)
print(f"Saved dense point cloud with {len(mesh_points)} points!")

Saved dense point cloud with 20000 points!
